In [ ]:
# Cell 1 — Clone / Update Repo
repo_url = "https://github.com/siddheshkadane01/Site-Reliability-Server/"
repo_dir = repo_url.rstrip("/").split("/")[-1]

import os
if os.path.exists(repo_dir):
    !git -C {repo_dir} pull origin sriram
else:
    !git clone -b sriram {repo_url}

%cd {repo_dir}

In [ ]:
# Cell 2 - All installs in one cell, correct order:
%pip install -r requirements.txt
%pip install -r requirements_rl.txt

# Pin TRL (important for GRPO stability)
%pip install trl==0.18.2 --quiet

!pip show trl | grep Version

In [ ]:
# Cell 3 - Start FastAPI server:
import subprocess, time, requests

log_file = open("server.log", "w")

server_process = subprocess.Popen(
    ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "7860"],
    stdout=log_file,
    stderr=log_file
)

time.sleep(12)  # increased for stability

try:
    res = requests.get("http://localhost:7860/health", timeout=5)
    print("✅ Server running" if res.status_code == 200 else f"❌ Unexpected status: {res.status_code}")
except Exception as e:
    print("❌ Server failed:", e)
    print(open("server.log").read())

# Optional debug preview
print(open("server.log").read()[:500])

In [ ]:
# Cell 4 - Wandb login using secret:
import wandb, os
from google.colab import userdata

os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

wandb.login()

from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])

print("✅ HuggingFace login successful")

In [ ]:
# Cell 5 - Train:
import os
os.environ["WANDB_PROJECT"] = "openenv-enterprise-grpo"
os.environ["WANDB_RUN_NAME"] = "grpo-qwen7b-final"
os.environ["WANDB_MODE"] = "online"

!python train_grpo.py \
    --epochs 3 \
    --num_generations 4 \
    --per_device_train_batch_size 2 \
    --dataset_size 32 \
    --temperature 1.0 \
    --top_p 0.92 \
    --lora_dropout 0.0 \
    --learning_rate 5e-6 \
    --env_url http://localhost:7860 \
    --model_name Qwen/Qwen2.5-7B-Instruct \
    --max_new_tokens 48 \
    --output_dir ./artifacts/grpo-qwen7b